# iNDI production - nucleus segmentation testing
Marianita completed image aquisition of the first 23 plates of data and everything has been copied to data/CARDPB2/iNDI\
Alena has been working on metadata aggregation and samplesheets\

## Important aquisition notes
Generally, imaging went well, but some wells from plates (and at least one whole plate) were aquired on different days, be sure to include aquisition dates in data metrics to test whether imaging date had any effect on phenotypes\
Marianita also noticed a registration issue with at least two of the imaging channels (LAMP1 and EEA1) resulting in a missalignment of several pixels. The cause of misalignment is not clear and we are uncertain how consistent it is - it might be worth it to run correlation of these channels across all images to see if there was a shift in correlation relative to time?

## Experimental metadata extraction

In [6]:
import matplotlib.pyplot as plt
from matplotlib.colors import LinearSegmentedColormap

# Define pseudocolor mapping rules - helpful to generate images on the fly
pseudocolor_map = {
    "DAPI": "blue",
    "Brightfield": "gray",
    "Alexa 488": "green",
    "Alexa 568": "red",
    "Alexa 647": "magenta"
}

mpl_colormaps = {
    "blue": LinearSegmentedColormap.from_list("black_blue", [(0, 0, 0), (0, 0, 1)]),
    "green": LinearSegmentedColormap.from_list("black_green", [(0, 0, 0), (0, 1, 0)]),
    "red": LinearSegmentedColormap.from_list("black_red", [(0, 0, 0), (1, 0, 0)]),
    "magenta": LinearSegmentedColormap.from_list("black_magenta", [(0, 0, 0), (1, 0, 1)]),
    "gray": LinearSegmentedColormap.from_list("black_gray", [(0, 0, 0), (1, 1, 1)])
}

In [11]:
import xml.etree.ElementTree as ET
from pathlib import Path
import pandas as pd

# 0c2035b2-6f9d-45b0-919b-94abbd4ad465
tree = ET.parse("/data/CARDPB2/iNDI/Production/0c2035b2-6f9d-45b0-919b-94abbd4ad465/0c2035b2-6f9d-45b0-919b.xml")
OF_dir = Path("/data/CARDPB2/iNDI/Production/0c2035b2-6f9d-45b0-919b-94abbd4ad465/images")
index_tree = ET.parse('/data/CARDPB2/iNDI/Production/0c2035b2-6f9d-45b0-919b-94abbd4ad465/images/index.xml')

root = tree.getroot()

# Namespace mapping
ns = {'h': '43B2A954-E3C3-47E1-B392-6635266B0DD3/HarmonyV7'} # I believe this is consistent between all experiments

# Find elements
measurement_id = root.find('h:MeasurementID', ns).text
date = root.find('h:Date', ns).text
plate = root.find('h:InitialPlateName', ns).text

print("Measurement ID:", measurement_id)
print("Date:", date)
print("Plate:", plate)

root = index_tree.getroot()

# Find elements
plate_id = root.find('.//h:PlateID', ns).text
x_res = float(root.find('.//h:ImageResolutionX', ns).text) * 1e6
y_res = float(root.find('.//h:ImageResolutionY', ns).text) * 1e6

print("Plate:", plate_id)
print(f"X resolution: {x_res} µm")
print(f"Y resolution: {y_res} µm")

channels = []

# Find the Map element that contains channel info entries
for map_el in root.findall(".//h:Map", ns):
    # Check first entry if it has a ChannelName child
    first_entry = map_el.find("h:Entry", ns)
    if first_entry is not None and first_entry.find("h:ChannelName", ns) is not None:
        # Found the Map with channel entries
        for entry in map_el.findall("h:Entry", ns):
            ch_id = entry.attrib.get("ChannelID")
            ch_name = entry.find("h:ChannelName", ns).text
            ch_type = entry.find("h:ChannelType", ns).text if entry.find("h:ChannelType", ns) is not None else None
            excitation = entry.find("h:MainExcitationWavelength", ns).text if entry.find("h:MainExcitationWavelength", ns) is not None else None
            emission = entry.find("h:MainEmissionWavelength", ns).text if entry.find("h:MainEmissionWavelength", ns) is not None else None

            channels.append({
                "ChannelID": int(ch_id) if ch_id is not None else None,
                "Channel_name": ch_name,
                "Type": ch_type,
                "Excitation_nm": excitation,
                "Emission_nm": emission
            })
        break  

# Convert to DataFrame
channel_df = pd.DataFrame(channels).sort_values('ChannelID').reset_index(drop=True)
print(channel_df)

channel_df["Pseudocolor"] = channel_df["Channel_name"].map(pseudocolor_map).fillna("gray")
channel_df["MPL_colormap"] = channel_df["Pseudocolor"].str.lower().map(mpl_colormaps)
channel_df["Measurement_ID"] = measurement_id
channel_df["Measurement_date"] = date
channel_df["Plate_ID"] = plate_id
channel_df["res_x"] = x_res
channel_df["res_y"] = y_res

print(channel_df)

Measurement ID: 0c2035b2-6f9d-45b0-919b-94abbd4ad465
Date: 2026-02-16T09:16:05.6991184-05:00
Plate: INDIMS0000
Plate: INDIMS0000
X resolution: 0.09418670872212731 µm
Y resolution: 0.09418670872212731 µm
   ChannelID Channel_name          Type Excitation_nm Emission_nm
0          1         DAPI  Fluorescence           375         456
1          2    Alexa 647  Fluorescence           640         706
2          3    Alexa 488  Fluorescence           488         522
3          4    Alexa 568  Fluorescence           561         599
   ChannelID Channel_name          Type Excitation_nm Emission_nm Pseudocolor  \
0          1         DAPI  Fluorescence           375         456        blue   
1          2    Alexa 647  Fluorescence           640         706     magenta   
2          3    Alexa 488  Fluorescence           488         522       green   
3          4    Alexa 568  Fluorescence           561         599         red   

                                        MPL_colormap  \
0  <m